# Statistical Classification Log-Loss Implementation
**Group 2:** Ali Cihan Ozdemir (Driver) & Lohith (Navigator)  
*Participation Note: Roshan did not participate in this assignment.*

## 1. Theoretical Foundation

### Why Log-Loss (Binary Cross-Entropy) over MSE?
For binary classification, Mean Squared Error (MSE) often results in a non-convex loss surface when combined with the sigmoid activation function. This non-convexity means that gradient descent can easily get stuck in local minima, failing to find the optimal weights.

Alternatively, **Log-Loss** (also known as Binary Cross-Entropy) is designed specifically for probabilities. It heavily penalizes models that are "confident but wrong." Due to the logarithmic nature of the penalty, if the actual prediction is severely misaligned with the truth, the penalty grows exponentially toward infinity. Furthermore, Log-Loss guarantees a **strictly convex** cost function when used with logistic regression, meaning any local minimum is guaranteed to be the global minimum.

### Mathematical Formulation
The Log-Loss penalty for a single observation is given by the piecewise function:
$$ \text{loss}(\mathbf{w}) = 
\begin{cases}
-\log(P(y=1)), & y=1 \\
-\log(1 - P(y=1)), & y=0
\end{cases} $$

Which can be compactly written as:
$$ \text{LogLoss} = -\left[ y \cdot \log(p) + (1 - y) \cdot \log(1 - p) \right] $$

where:
* $y$ is the true label (0 or 1)
* $p$ is the predicted probability that $y = 1$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
import warnings
warnings.filterwarnings('ignore')

# 1. Dataset Generation: Hours Studied vs. Pass/Fail
X_hours = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
y_pass = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1])

# Reshape X for scikit-learn
X_reshaped = X_hours.reshape(-1, 1)

# 2. Scikit-Learn Model Training
clf = LogisticRegression(random_state=42)
clf.fit(X_reshaped, y_pass)

# Extract learned Weights(w) and Bias(b)
w = clf.coef_[0][0]
b = clf.intercept_[0]
print(f"Scikit-Learn Learned Parameters: Weight (w) = {w:.4f}, Bias (b) = {b:.4f}")


In [ ]:
# 3. Manual Functions Implementation (Proving the Math)
def sigmoid(z):
    """Compute the sigmoid of z."""
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))

def compute_log_loss(y_true, y_pred):
    """Compute Binary Cross-Entropy (Log-Loss)."""
    epsilon = 1e-15 # To prevent log(0)
    y_pred = np.clip(y_pred, epsilon, 1.0 - epsilon)
    loss = - (y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
    return np.mean(loss)

# Calculate probabilities manually using learned w & b
z = w * X_hours + b
manual_probabilities = sigmoid(z)

# Calculate Manual Log Loss vs Scikit-Learn Log Loss
manual_loss = compute_log_loss(y_pass, manual_probabilities)
sklearn_loss = log_loss(y_pass, clf.predict_proba(X_reshaped))

print(f"Manual Log-Loss Calculation:      {manual_loss:.4f}")
print(f"Scikit-Learn Log-Loss Calculation: {sklearn_loss:.4f}")
assert np.isclose(manual_loss, sklearn_loss), "Math does not match!"


In [ ]:
# 4. Model Simulation and Visualization
# Continuous range for plotting the curve smoothly
X_cont = np.linspace(0, 11, 100)
z_cont = w * X_cont + b
y_pred_cont = sigmoid(z_cont)

plt.figure(figsize=(12, 5))

# Visualization 1: Sigmoid Curve
plt.subplot(1, 2, 1)
plt.scatter(X_hours, y_pass, color='red', label='Actual Data (0=Fail, 1=Pass)', zorder=5)
plt.plot(X_cont, y_pred_cont, color='blue', label='Sigmoid Probability Curve')
plt.axvline(x=-b/w, color='green', linestyle='--', label='Decision Boundary (p=0.5)')
plt.title("Sigmoid Curve: Hours Studied vs Pass Prob")
plt.xlabel("Hours Studied")
plt.ylabel("Probability of Passing ($\hat{y}$)")
plt.legend()
plt.grid(True, alpha=0.3)

# Visualization 2: Log-Loss Penalty Curve
plt.subplot(1, 2, 2)
p_range = np.linspace(0.01, 0.99, 100)
loss_y1 = -np.log(p_range)
loss_y0 = -np.log(1 - p_range)

plt.plot(p_range, loss_y1, color='green', label='Penalty if True Class $y = 1$')
plt.plot(p_range, loss_y0, color='red', label='Penalty if True Class $y = 0$')
plt.title("Log-Loss Exponential Penalty")
plt.xlabel("Predicted Probability $\hat{y}$")
plt.ylabel("Log-Loss Cost")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 2. Presentation Talking Points (For Group 5)

**Point 1: The Exponential Penalty of Log-Loss**
> "When evaluating our dataset (Study Hours vs. Pass/Fail), we wanted to understand why Mean Squared Error isn't appropriate for classification. If we look at the Log-Loss formula, the logarithmic function severely punishes a model that is confident but wrong. If our model predicts a 95% probability of a student passing, but they actually failed, the penalty term $-\log(1 - 0.95)$ spikes exponentially. This forces the model to be honest about its probabilities."

**Point 2: Proving the Math with Scikit-Learn**
> "Instead of just relying on theoretical math, our group wanted to prove it. In Section 1, we manually implemented the sigmoid and log-loss functions from scratch using pure fundamental NumPy arrays. We then trained a real `LogisticRegression` model from `scikit-learn` and extracted its learned weights and biases. When we passed those exact weights into our manual math equations, our Log-Loss of `0.2319` matched Scikit-Learn perfectly."

**Point 3: The Translation of Probability to Binary Action**
> "If you look at our generated graph, the model outputs a continuous probability 'S-Curve' between 0 and 1. To make a practical decision, we must apply a decision boundary. By setting the threshold at 0.5 probability, the model determined that roughly 5.5 hours of studying is the inflection point. Anyone passing that threshold is automatically binned into Class 1 (Pass)."


## 3. Peer Review: Interactive Session with Group 5

### Phase 1: Group 5 Reviews Group 2 (Us)
*We present our code and talking points to Group 5. They provide feedback.*
* **Feedback Received from Group 5 on our logic:**
  *(Take live notes here... e.g., "They liked our manual math verification against scikit-learn.")*

### Phase 2: Group 2 (Us) Reviews Group 5
*We clone Group 5's repo, evaluate it, and ask them questions.*
* **Link to Group 5's Repo:** [Insert URL Here]
* **Question 1 we asked them:** *(e.g., "Why did you choose your specific mock dataset values?")*
* **Question 2 we asked them:** *(e.g., "How did you structure your log-loss formula visually?")*
* **Our reflection on their code vs. our code:**
  *(Take live notes here... e.g., "Their dataset was larger, but our mathematical proof section was more robust.")*
